# **Inference/Testing**

### **Import Library**

Melakukan import library yang dibutuhkan untuk submission.

In [1]:
import requests
import pandas as pd
import tensorflow as tf
import json
import base64

### **Load Dataset**

Membaca dataset bernama "Corona_NLP.csv" berada dalam folder "data" menggunakan library pandas kemudian disimpan ke dalam variabel data. Lanjut menampilkan ringkasan informasi dari DataFrame.

In [ ]:
data = pd.read_csv('data\Corona_NLP.csv')
data

,OriginalTweet,Sentiment
0,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,advice Talk to your neighbours family to excha...,Positive
2,Coronavirus Australia: Woolworths to give elde...,Positive
3,My food stock is not the only one which is emp...,Positive
4,"Me, ready to go at supermarket during the #COV...",Extremely Negative
...,...,...
41152,Airline pilots offering to stock supermarket s...,Neutral
41153,Response to complaint not provided citing COVI...,Extremely Negative
41154,You know itÂs getting tough when @KameronWild...,Positive
41155,Is it wrong that the smell of hand sanitizer i...,Neutral


Menampilkan data OriginalTweet di data

In [4]:
data['OriginalTweet']

0        @MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...
1        advice Talk to your neighbours family to excha...
2        Coronavirus Australia: Woolworths to give elde...
3        My food stock is not the only one which is emp...
4        Me, ready to go at supermarket during the #COV...
                               ...                        
41152    Airline pilots offering to stock supermarket s...
41153    Response to complaint not provided citing COVI...
41154    You know itÂs getting tough when @KameronWild...
41155    Is it wrong that the smell of hand sanitizer i...
41156    @TartiiCat Well new/used Rift S are going for ...
Name: OriginalTweet, Length: 41157, dtype: object

### **Mengambil data yang akan ditest**

Mengambil data dari text ke 1 dan menginisialisasikan ke dalam text_test untuk proses test

In [13]:
OriginalTweet_test = data['OriginalTweet'][1]

OriginalTweet_test

'advice Talk to your neighbours family to exchange phone numbers create contact list with phone numbers of neighbours schools employer chemist GP set up online shopping accounts if poss adequate supplies of regular meds but not over order'

### **Prediksi**

Mempersiapkan data input untuk prediksi menggunakan model yang telah dilatih. Kemudian mengirimkan permintaan prediksi ke endpoint model TensorFlow Serving.

Respons dari server kemudian diproses untuk mengambil nilai prediksi dan menentukan hasilnya. Jika nilai prediksi lebih besar dari 0.5, hasilnya adalah sentiment tweet. Jika tidak ada prediksi ditemukan dalam respons, maka hasil akan menampilkan pesan kesalahan.

In [14]:
from typing import Literal


def prepare_json(OriginalTweet):
    feature_spec = {
        "OriginalTweet": tf.train.Feature(bytes_list=tf.train.BytesList(value=[bytes(OriginalTweet, "utf-8")])),
    }
    
    example = tf.train.Example(
        features=tf.train.Features(feature=feature_spec)
    ).SerializeToString()
    
    result = [
        {
            "examples": {
                "b64": base64.b64encode(example).decode()
            }
        }
    ]
    
    return json.dumps({
        "signature_name": "serving_default",
        "instances": result
    })

json_data = prepare_json(OriginalTweet_test)
    
endpoint = "http://localhost:8080/v1/models/coronavirus-prediction-model:predict"
response = requests.post(endpoint, data=json_data)

result = response.json()
print(json.dumps(result, indent=2))

{
  "predictions": [
    [
      0.877931297,
      0.0493978113,
      0.0250861719,
      0.047261171,
      0.000323498738
    ]
  ]
}


In [15]:
predictions = result.get("predictions", [])
if predictions:
    probability_vector = predictions[0]
    predicted_index = max(range(len(probability_vector)), key=lambda idx: probability_vector[idx])
    print("Probability vector:", probability_vector)
    print("Predicted class index:", predicted_index)
else:
    print("Tidak ada prediksi yang dikembalikan endpoint serving.")

Probability vector: [0.877931297, 0.0493978113, 0.0250861719, 0.047261171, 0.000323498738]
Predicted class index: 0
